In [0]:
###Document Structure

from langchain_core.documents import Document

In [0]:
%sh
pip install langchain_community
pip install pymupdf

Read PDF File From Here

In [0]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

dir_loader=DirectoryLoader(
    "./Data/PDF_Files",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents = dir_loader.load()

print(f"Total pages loaded: {len(pdf_documents)}")
pdf_documents

In [0]:
# Check total documents loaded
print(f"Total documents (pages) loaded: {len(pdf_documents)}")
print(f"\nUnique PDF files loaded:")

# Get unique file sources
file_sources = set()
for doc in pdf_documents:
    file_sources.add(doc.metadata.get('source', 'Unknown'))

for i, source in enumerate(sorted(file_sources), 1):
    file_name = source.split('/')[-1]
    page_count = sum(1 for doc in pdf_documents if doc.metadata.get('source') == source)
    print(f"{i}. {file_name}: {page_count} pages")

print(f"\nTotal unique PDF files: {len(file_sources)}")

In [0]:
type(pdf_documents[0])

PDF Loader Part Starting From Here

In [0]:
%pip install langchain-community langchain 'langchain-protocol<0.0.18' chromadb
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [0]:
%sh 
pip install chromadb
pip install sentence_transformers
pip install pypdf

In [0]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Update with your workspace path
all_pdf_documents = process_all_pdfs(
    "/Workspace/Users/draxop7536@gmail.com/RAG_1/Data/PDF_Files"
)

print(all_pdf_documents[:2])

Split PDF into smaller chunks

In [0]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [0]:
import re
from langchain_core.documents import Document

def split_documents_by_article(documents):
    """
    Split Constitution documents into chunks based on Article boundaries.
    Each Article becomes one chunk.
    """

    split_docs = []

    article_pattern = r'(?=Article\s+\d+[A-Z]?)'

    for doc in documents:
        text = doc.page_content

        # Split while retaining "Article X" at the beginning
        articles = re.split(article_pattern, text)

        for article in articles:
            article = article.strip()

            if not article:
                continue

            # Extract article number
            article_match = re.search(r'Article\s+(\d+[A-Z]?)', article)

            metadata = doc.metadata.copy()

            if article_match:
                metadata["article"] = article_match.group(1)

            split_docs.append(
                Document(
                    page_content=article,
                    metadata=metadata
                )
            )

    print(f"Split {len(documents)} documents into {len(split_docs)} article chunks")

    if split_docs:
        print("\nExample chunk:")
        print(split_docs[0].page_content[:500])
        print(split_docs[0].metadata)

    return split_docs

In [0]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents_by_article1(documents):
    
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    split_docs = []

    for doc in documents:

        articles = re.split(
            r'(?=Article\s+\d+[A-Z]?)',
            doc.page_content
        )

        for article_text in articles:

            article_text = article_text.strip()

            if not article_text:
                continue

            article_match = re.search(
                r'Article\s+(\d+[A-Z]?)',
                article_text
            )

            metadata = doc.metadata.copy()

            if article_match:
                metadata["article"] = article_match.group(1)

            # Small article -> single chunk
            if len(article_text) <= 1000:

                split_docs.append(
                    Document(
                        page_content=article_text,
                        metadata=metadata
                    )
                )

            # Large article -> split further
            else:

                child_docs = child_splitter.create_documents(
                    [article_text],
                    metadatas=[metadata]
                )

                split_docs.extend(child_docs)

    print(
        f"Split {len(documents)} documents into {len(split_docs)} chunks"
    )

    return split_docs

In [0]:
chunks=split_documents_by_article1(all_pdf_documents)
chunks

In [0]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

Embadding

In [0]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "BAAI/bge-base-en-v1.5"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager




















# class EmbeddingManager:
#     """Handles document embedding generation using SentenceTransformer"""
    
#     def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
#         """
#         Initialize the embedding manager
        
#         Args:
#             model_name: HuggingFace model name for sentence embeddings
#         """
#         self.model_name = model_name
#         self.model = None
#         self._load_model()

#     def _load_model(self):
#         """Load the SentenceTransformer model"""
#         try:
#             print(f"Loading embedding model: {self.model_name}")
#             self.model = SentenceTransformer(self.model_name)
#             print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
#         except Exception as e:
#             print(f"Error loading model {self.model_name}: {e}")
#             raise

#     def generate_embeddings(self, texts: List[str]) -> np.ndarray:
#         """
#         Generate embeddings for a list of texts
        
#         Args:
#             texts: List of text strings to embed
            
#         Returns:
#             numpy array of embeddings with shape (len(texts), embedding_dim)
#         """
#         if not self.model:
#             raise ValueError("Model not loaded")
        
#         print(f"Generating embeddings for {len(texts)} texts...")
#         embeddings = self.model.encode(texts, show_progress_bar=True)
#         print(f"Generated embeddings with shape: {embeddings.shape}")
#         return embeddings


# ## initialize the embedding manager

# embedding_manager=EmbeddingManager()
# embedding_manager

In [0]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./Data/Vector_Store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [0]:
chunks

In [0]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

## Store in the vector database
# Use a fresh directory path to avoid ChromaDB lock/corruption issues
import uuid
import shutil

# Create a unique directory name to ensure clean state
vector_store_path = f"/Workspace/Users/draxop7536@gmail.com/RAG_1/Data/Vector_Store_{uuid.uuid4().hex[:8]}"

vectorstore = VectorStore(persist_directory=vector_store_path)
vectorstore.add_documents(chunks,embeddings)



# ### Convert the text to embeddings
# texts=[doc.page_content for doc in chunks]

# ## Generate the Embeddings

# embeddings=embedding_manager.generate_embeddings(texts)

# ##store int he vector dtaabase
# vectorstore.add_documents(chunks,embeddings)

In [0]:
# Point to your EXISTING vector store directory
vector_store_path = "/Workspace/Users/draxop7536@gmail.com/RAG_1/Data/Vector_Store_c905e30d"
vectorstore = VectorStore(persist_directory=vector_store_path)

Temporary data retrival testing from vectorstore

In [0]:
# Query
query_text = "In which year this new constitution was adopted?"

# Generate query embedding using the same embedding model
query_embedding = embedding_manager.generate_embeddings([query_text])[0]

results = vectorstore.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2
)

# Format the output
print(f"Query: {query_text}")
print("=" * 80)
print()

if results['documents'] and len(results['documents'][0]) > 0:
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"Result {i+1}:")
        print(f"Source: {metadata.get('source_file', 'N/A')}")
        print(f"Page: {metadata.get('page', 'N/A') + 1}")
        print(f"Relevance Score: {1 - distance:.4f}")
        print()
        print("Content:")
        print(doc)
        print()
        print("-" * 80)
        print()
else:
    print("No results found.")

Retrival Logic

In [0]:
from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: Search query
            top_k: Number of documents to retrieve

        Returns:
            List of retrieved documents
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")

        try:
            # Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            # Query ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and len(results["documents"][0]) > 0:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for rank, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances),
                    start=1
                ):

                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "rank": rank
                    })

                    print(f"Rank: {rank} | Distance: {distance:.4f}")

                print(f"Retrieved {len(retrieved_docs)} documents")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [0]:
rag_retriever

In [0]:
# Query the RAG system (recreate retriever to use the proper RAGRetriever class)
query_text = "In which year this new constitution was adopted?"
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
results = rag_retriever.retrieve(query_text, top_k=5)


# Format the output nicely
print("\n" + "=" * 80)
print(f"Query: {query_text}")
print("=" * 80)
print()

if results:
    for result in results:
        print(f"Rank {result['rank']}:")
        print(f"Source: {result['metadata'].get('source_file', 'N/A')}")
        print(f"Page: {result['metadata'].get('page', 'N/A') + 1}")
        print(f"Relevance Score: {1 - result['distance']:.4f}")
        print()
        print("Content:")
        print(result['content'])
        print()
        print("-" * 80)
        print()
else:
    print("No results found.")

Integrating VectorDB Context pipeline With LLM Output

In [0]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY1"))

Same as in GITHUB

In [0]:
%pip install -q langchain-groq langchain
dbutils.library.restartPython()

In [0]:
%pip install -q langchain-groq

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [0]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY1")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY1 environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [0]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY1"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY1 environment variable to use the LLM.")
    groq_llm = None

In [0]:
rag_retriever.retrieve("In which year this new constitution was adopted?")

In [0]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY1")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [0]:
answer=rag_simple("In which year constitution was adopted?",rag_retriever,llm)
print(answer)

In [0]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Filter by minimum score (distance-based: lower distance = higher similarity)
    filtered_results = [doc for doc in results if (1 - doc['distance']) >= min_score]
    
    if not filtered_results:
        return {'answer': 'No relevant context found meeting the minimum score threshold.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in filtered_results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': 1 - doc['distance'],
        'preview': doc['content'][:300] + '...'
    } for doc in filtered_results]
    confidence = max([1 - doc['distance'] for doc in filtered_results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("In which year this new constitution was adopted?", rag_retriever, llm, top_k=3, min_score=0.05, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

## Testing the Modular RAG System
Now we can import and use the rag_core module

In [0]:
# Import the module
import sys
sys.path.insert(0, '/Workspace/Users/draxop7536@gmail.com')

from rag_core import initialize_rag_system
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize the RAG system with one line!
rag_pipeline, retriever, embedding_mgr, vectorstore = initialize_rag_system(
    vector_store_path="/Workspace/Users/draxop7536@gmail.com/RAG_1/Data/Vector_Store_3f5994b2",
    groq_api_key=os.getenv("GROQ_API_KEY1"),
    model_name="llama-3.1-8b-instant"
)

print("✅ RAG System initialized successfully!")

In [0]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            # Filter by minimum score (distance-based: lower distance = higher similarity)
            filtered_results = [doc for doc in results if (1 - doc['distance']) >= min_score]
            
            if not filtered_results:
                answer = "No relevant context found meeting the minimum score threshold."
                sources = []
                context = ""
            else:
                context = "\n\n".join([doc['content'] for doc in filtered_results])
                sources = [{
                    'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                    'page': doc['metadata'].get('page', 'unknown'),
                    'score': 1 - doc['distance'],
                    'preview': doc['content'][:120] + '...'
                } for doc in filtered_results]
                results = filtered_results
                
                # Generate answer only if we have context
                # Streaming answer simulation
                prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
                if stream:
                    print("Streaming answer:")
                    for i in range(0, len(prompt), 80):
                        print(prompt[i:i+80], end='', flush=True)
                        time.sleep(0.05)
                    print()
                response = self.llm.invoke([prompt])
                answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("The Constituent Assembly of India originally had how many members?", top_k=3, min_score=0.05, stream=False, summarize=True)

print("\n" + "="*80)
print("ADVANCED RAG PIPELINE RESULTS")
print("="*80)
print("\nQuestion:", result['question'])
print("\nAnswer:", result['answer'])
print("\nSources:")
for i, src in enumerate(result['sources'], 1):
    print(f"  [{i}] {src['source']} (page {src['page']}) - Score: {src['score']:.4f}")
print("\nSummary:", result['summary'])
print("\nQuery History Count:", len(result['history']))

In [0]:
import json
from datetime import datetime
from typing import List, Dict, Any, Optional

class HistoricalRAGPipeline(AdvancedRAGPipeline):
    """Extended RAG Pipeline with query history persistence and search"""
    
    def __init__(self, retriever, llm, history_file: str = "/Workspace/Users/draxop7536@gmail.com/RAG_1/Data/query_history.json"):
        super().__init__(retriever, llm)
        self.history_file = history_file
        self.load_history()
    
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        """Override query to add timestamp and handle deduplication"""
        # Check if this exact question already exists in history
        existing_idx = None
        for idx, entry in enumerate(self.history):
            if entry['question'].strip().lower() == question.strip().lower():
                existing_idx = idx
                break
        
        # If duplicate found, remove it temporarily (we'll add updated version)
        if existing_idx is not None:
            old_entry = self.history.pop(existing_idx)
            query_count = old_entry.get('query_count', 1) + 1
            first_asked = old_entry.get('first_asked', old_entry.get('timestamp', datetime.now().isoformat()))
            print(f"⚠️  Duplicate query detected! This question was asked {query_count} times.")
        else:
            query_count = 1
            first_asked = datetime.now().isoformat()
        
        # Execute the query
        result = super().query(question, top_k, min_score, stream, summarize)
        
        # Update the last history entry with enhanced metadata
        if self.history:
            self.history[-1]['timestamp'] = datetime.now().isoformat()
            self.history[-1]['query_count'] = query_count
            self.history[-1]['first_asked'] = first_asked
            self.history[-1]['last_asked'] = datetime.now().isoformat()
        
        # Auto-save after each query
        self.save_history()
        
        return result
    
    def save_history(self):
        """Save query history to JSON file"""
        try:
            # Create directory if it doesn't exist
            import os
            os.makedirs(os.path.dirname(self.history_file), exist_ok=True)
            
            with open(self.history_file, 'w') as f:
                json.dump(self.history, f, indent=2)
            print(f"History saved to {self.history_file}")
        except Exception as e:
            print(f"Error saving history: {e}")
    
    def load_history(self):
        """Load query history from JSON file"""
        try:
            with open(self.history_file, 'r') as f:
                self.history = json.load(f)
            print(f"Loaded {len(self.history)} historical queries from {self.history_file}")
        except FileNotFoundError:
            print(f"No existing history file found. Starting fresh.")
            self.history = []
        except Exception as e:
            print(f"Error loading history: {e}")
            self.history = []
    
    def search_history(self, search_query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        """Search through historical queries using embedding similarity"""
        if not self.history:
            print("No query history available.")
            return []
        
        print(f"Searching through {len(self.history)} historical queries...")
        
        # Generate embedding for search query
        search_embedding = self.retriever.embedding_manager.generate_embeddings([search_query])[0]
        
        # Generate embeddings for all historical questions
        historical_questions = [entry['question'] for entry in self.history]
        historical_embeddings = self.retriever.embedding_manager.generate_embeddings(historical_questions)
        
        # Calculate cosine similarity
        from numpy.linalg import norm
        similarities = []
        for i, hist_emb in enumerate(historical_embeddings):
            similarity = np.dot(search_embedding, hist_emb) / (norm(search_embedding) * norm(hist_emb))
            similarities.append((i, similarity))
        
        # Sort by similarity (highest first)
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        # Return top_k results
        results = []
        for i, sim in similarities[:top_k]:
            entry = self.history[i].copy()
            entry['similarity_score'] = sim
            results.append(entry)
        
        return results
    
    def get_history_stats(self) -> Dict[str, Any]:
        """Get statistics about query history"""
        if not self.history:
            return {'total_queries': 0}
        
        total_asks = sum(entry.get('query_count', 1) for entry in self.history)
        duplicates = sum(1 for entry in self.history if entry.get('query_count', 1) > 1)
        
        return {
            'unique_queries': len(self.history),
            'total_asks': total_asks,
            'queries_with_duplicates': duplicates,
            'first_query': self.history[0].get('first_asked', self.history[0].get('timestamp', 'N/A')),
            'last_query': self.history[-1].get('last_asked', self.history[-1].get('timestamp', 'N/A')),
            'unique_sources': len(set(
                src['source'] 
                for entry in self.history 
                for src in entry.get('sources', [])
            ))
        }
    
    def get_frequent_queries(self, top_k: int = 5) -> List[Dict[str, Any]]:
        """Get the most frequently asked queries"""
        if not self.history:
            print("No query history available.")
            return []
        
        # Sort by query_count descending
        sorted_history = sorted(
            self.history, 
            key=lambda x: x.get('query_count', 1), 
            reverse=True
        )
        
        return sorted_history[:top_k]
    
    def clear_history(self):
        """Clear all query history"""
        self.history = []
        self.save_history()
        print("Query history cleared.")

# Initialize the historical RAG pipeline
hist_rag = HistoricalRAGPipeline(rag_retriever, llm)

print("\n" + "="*80)
print("HISTORICAL RAG PIPELINE INITIALIZED")
print("="*80)
print(f"\nHistory file: {hist_rag.history_file}")
print(f"Loaded queries: {len(hist_rag.history)}")

In [0]:
# --- DEMO: Using Historical RAG Pipeline ---

# 1. Ask a new question (will be saved to history)
print("\n" + "="*80)
print("ASKING NEW QUESTION")
print("="*80)

question1 = "In which year was this new constitution adopted?"
result1 = hist_rag.query(question1, top_k=3, min_score=0.05, summarize=True)

print(f"\nQuestion: {result1['question']}")
print(f"\nAnswer: {result1['answer']}")
if result1['summary']:
    print(f"\nSummary: {result1['summary']}")

# 2. Ask another question
print("\n" + "="*80)
print("ASKING ANOTHER QUESTION")
print("="*80)

question2 = "How many members were in the Constituent Assembly?"
result2 = hist_rag.query(question2, top_k=3, min_score=0.05, summarize=True)

print(f"\nQuestion: {result2['question']}")
print(f"\nAnswer: {result2['answer']}")
if result2['summary']:
    print(f"\nSummary: {result2['summary']}")

# 3. View history statistics
print("\n" + "="*80)
print("QUERY HISTORY STATISTICS")
print("="*80)

stats = hist_rag.get_history_stats()
for key, value in stats.items():
    print(f"{key}: {value}")

# 4. Search through historical queries
print("\n" + "="*80)
print("SEARCHING HISTORICAL QUERIES")
print("="*80)

search_query = "constitution adoption date"
print(f"\nSearch Query: '{search_query}'")
print("\nTop 3 Similar Historical Queries:\n")

historical_results = hist_rag.search_history(search_query, top_k=3)

for i, result in enumerate(historical_results, 1):
    print(f"[{i}] Question: {result['question']}")
    print(f"    Similarity: {result['similarity_score']:.4f}")
    print(f"    Timestamp: {result.get('timestamp', 'N/A')}")
    print(f"    Answer Preview: {result['answer'][:150]}...")
    print()

print("="*80)

In [0]:
# Test with a completely different query (unrelated to previous ones)
print("\n" + "="*80)
print("TESTING WITH COMPLETELY DIFFERENT QUERY")
print("="*80)

# This query is totally different from "constitution adoption" or "Constituent Assembly"
test_query = "What are the provisions related to education?"
print(f"\nAsking: '{test_query}'\n")

result = hist_rag.query(test_query, top_k=3, min_score=0.05, summarize=True)

print(f"\nQuestion: {result['question']}")
print(f"\nAnswer: {result['answer']}")
if result['summary']:
    print(f"\nSummary: {result['summary']}")

print("\n" + "="*80)
print("RESULT: hist_rag successfully answered a completely different query!")
print("="*80)

# Now let's check the history
print(f"\nTotal queries in history: {len(hist_rag.history)}")
print("\nAll historical queries:")
for i, entry in enumerate(hist_rag.history, 1):
    print(f"{i}. {entry['question']}")

In [0]:
# --- DEMO: Testing Deduplication Feature ---

print("\n" + "="*80)
print("TESTING DEDUPLICATION")
print("="*80)

# Clear history first for a clean test
hist_rag.clear_history()

# Ask the same question 3 times
test_question = "What are the fundamental rights in the constitution?"

print(f"\nAsking the same question 3 times: '{test_question}'\n")

for i in range(1, 4):
    print(f"\n--- Attempt {i} ---")
    result = hist_rag.query(test_question, top_k=3, min_score=0.05, summarize=False)
    print(f"Answer preview: {result['answer'][:100]}...")

# Check history stats
print("\n" + "="*80)
print("HISTORY STATISTICS AFTER DEDUPLICATION")
print("="*80)

stats = hist_rag.get_history_stats()
for key, value in stats.items():
    print(f"{key}: {value}")

print("\n" + "="*80)
print("DETAILED HISTORY")
print("="*80)

for i, entry in enumerate(hist_rag.history, 1):
    print(f"\n[{i}] Question: {entry['question']}")
    print(f"    Query Count: {entry.get('query_count', 1)}")
    print(f"    First Asked: {entry.get('first_asked', 'N/A')}")
    print(f"    Last Asked: {entry.get('last_asked', 'N/A')}")

print("\n" + "="*80)
print("RESULT: Only 1 unique question stored, with query_count = 3!")
print("="*80)

In [0]:
# --- DEMO: View Most Frequently Asked Questions ---

print("\n" + "="*80)
print("MOST FREQUENTLY ASKED QUESTIONS")
print("="*80)

# Add a few more diverse questions to make it interesting
questions = [
    "What are the fundamental rights in the constitution?",  # Will increment existing
    "How many articles are in the constitution?",
    "What are the fundamental rights in the constitution?",  # Another duplicate
    "What is the role of the President?",
    "How many articles are in the constitution?",  # Another duplicate
]

print("\nAdding more queries to test frequency tracking...\n")
for q in questions:
    hist_rag.query(q, top_k=2, min_score=0.05, summarize=False)

print("\n" + "="*80)
print("TOP 5 FREQUENTLY ASKED QUESTIONS")
print("="*80 + "\n")

frequent = hist_rag.get_frequent_queries(top_k=5)

for i, entry in enumerate(frequent, 1):
    print(f"[{i}] Asked {entry.get('query_count', 1)} times")
    print(f"    Question: {entry['question']}")
    print(f"    First: {entry.get('first_asked', 'N/A')[:19]}")
    print(f"    Last: {entry.get('last_asked', 'N/A')[:19]}")
    print()

print("="*80)
print(f"\nTotal Unique Questions: {len(hist_rag.history)}")
print(f"Total Times Asked: {sum(e.get('query_count', 1) for e in hist_rag.history)}")